# EDA, Data Quality e ML Readiness — Previsão de Vendas

Este notebook é organizado como um diagnóstico de riscos antes da modelagem:

**1. Data Understanding → 2. Data Quality → 3. Analytical EDA → 4. ML Readiness**

O objetivo não é apenas visualizar os dados, mas responder quatro perguntas:
- **O que existe nos dados?**
- **Posso confiar neles?**
- **Como eles se comportam?**
- **Posso usá-los em ML sem introduzir vieses, leakage ou fragilidade?**

> Regra do projeto: detectar problemas não significa corrigir ou excluir automaticamente. Toda decisão deve ser tecnicamente justificável e, quando necessário, validada com o contexto de negócio.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from gb_ml.dq import (
    cold_start_report,
    correlation_leakage_candidates,
    feature_availability_audit,
    join_explosion_check,
    quality_gate,
    rare_category_report,
    run_quality_checks,
    temporal_drift_report,
)
from gb_ml.io import sample_table, table_metadata
from gb_ml.profiling import (
    outlier_report,
    plot_correlation_heatmap,
    plot_missingness_over_time,
    plot_numeric_distributions,
    plot_outlier_boxplots,
    plot_outlier_rate,
    plot_seasonality,
    plot_target_time_series,
    profile_dataframe,
    temporal_summary,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "tabelas" / "exemplo_vendas.yaml"

PROJECT_ID = "gms-prod-01"
TABLE_ID = "silver_gb.fact_vendas"
SAMPLE_LIMIT = 100_000

In [3]:
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

TARGET = config.get("target")
DATE_COL = config.get("date_column")
ENTITY_KEYS = config.get("entity_keys", [])

display(config)

{'table_name': 'exemplo_vendas',
 'target': 'vendas',
 'date_column': 'data',
 'grain': ['data', 'produto_id', 'loja_id'],
 'entity_keys': ['produto_id', 'loja_id'],
 'defaults': {'warn_null_pct': 0.05,
  'error_null_pct': 0.3,
  'warn_blank_pct': 0.01},
 'columns': {'data': {'type': 'datetime',
   'nullable': False,
   'date_no_future': True,
   'future_severity': 'error'},
  'produto_id': {'type': 'string', 'nullable': False},
  'loja_id': {'type': 'string', 'nullable': False},
  'vendas': {'type': 'numeric',
   'nullable': False,
   'min': 0,
   'bounds_severity': 'error'},
  'preco': {'type': 'numeric',
   'nullable': True,
   'min': 0,
   'bounds_severity': 'warn'}},
 'feature_availability': {'produto_id': {'available_at_prediction': True,
   'detail': 'Conhecido antes do horizonte de previsão.'},
  'loja_id': {'available_at_prediction': True,
   'detail': 'Conhecido antes do horizonte de previsão.'},
  'preco': {'available_at_prediction': True,
   'detail': 'Assumindo preço plane

# 1. Data Understanding

Antes de procurar anomalias, entendemos **fonte, schema, volume, período coberto, granularidade e significado das colunas**.

Para tabelas grandes no BigQuery, o princípio é: **agregar e filtrar no warehouse antes de trazer dados ao Pandas**.

## 1.1 Data discovery e metadados

In [4]:
if TABLE_ID:
    metadata, schema = table_metadata(TABLE_ID, project_id=PROJECT_ID)
    display(pd.DataFrame([metadata]))
    display(schema)
else:
    print("TABLE_ID ainda não configurado — usando dataset sintético de demonstração.")

,table_id,num_rows,num_bytes,created,modified,partitioning_type,partition_field,clustering_fields
0,gms-prod-01.silver_gb.fact_vendas,104000,5191520,2026-06-15 13:00:21.645000+00:00,2026-06-18 05:24:45.553000+00:00,TimePartitioning,data_semana,[sku_id]


,name,field_type,mode,description
0,data_semana,DATE,NULLABLE,Segunda-feira da semana de referência (DATE). ...
1,sku_id,STRING,NULLABLE,Produto vendido (FK dim_produtos).
2,vendas_previsao,INTEGER,NULLABLE,Previsão de vendas da semana (unidades).
3,vendas_real,INTEGER,NULLABLE,Vendas realizadas (unidades). NULL quando o re...
4,data_ingestao,TIMESTAMP,NULLABLE,Timestamp de ingestão na origem (bronze). Crit...
5,data_processado,TIMESTAMP,NULLABLE,Timestamp do run dbt (run_started_at) que mate...


## 1.2 Carga controlada

O dataset sintético contém problemas deliberados para exercitar o pipeline:
- duplicidade no grão;
- venda negativa;
- outlier extremo;
- missingness concentrada no período recente;
- produto novo com pouco histórico;
- mudança de distribuição de preço;
- feature que replica o target e representa leakage.

No case real, este bloco será substituído pela leitura das tabelas reais.

In [5]:
if TABLE_ID:
    df = sample_table(
        TABLE_ID,
        project_id=PROJECT_ID,
        limit=SAMPLE_LIMIT,
    )
else:
    rng = np.random.default_rng(42)
    dates = pd.date_range("2025-01-01", periods=180, freq="D")
    products = ["A", "B", "C"]
    stores = ["L01", "L02"]
    category_map = {"A": "Perfumaria", "B": "Maquiagem", "C": "Lançamento"}

    rows = []
    for date in dates:
        for product in products:
            if product == "C" and date < dates[-25]:
                continue

            for store in stores:
                base = {"A": 110, "B": 80, "C": 55}[product]
                store_effect = 1.10 if store == "L01" else 0.95
                weekend_effect = 1.25 if date.dayofweek >= 5 else 1.0
                promo = int(rng.random() < 0.12)
                promo_effect = 1.35 if promo else 1.0

                price = {"A": 30, "B": 22, "C": 27}[product] + rng.normal(0, 1.5)

                if date >= dates[-45]:
                    price += 5

                stockout = int(rng.random() < 0.03)
                expected_sales = base * store_effect * weekend_effect * promo_effect
                sales = max(0, rng.normal(expected_sales, 12))

                if stockout:
                    sales = 0

                rows.append({
                    "data": date,
                    "produto_id": product,
                    "loja_id": store,
                    "categoria": category_map[product],
                    "promocao": promo,
                    "ruptura_estoque": stockout,
                    "preco": round(price, 2),
                    "vendas": round(sales, 0),
                })

    df = pd.DataFrame(rows)
    df["vendas_confirmadas"] = df["vendas"]

    df.loc[df.index[10], "vendas"] = 2500
    df.loc[df.index[25], "vendas"] = -10

    recent_mask = (
        (df["data"] >= dates[-20])
        & (df["produto_id"] == "B")
        & (df["loja_id"] == "L02")
    )
    df.loc[recent_mask, "preco"] = np.nan

    df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

display(df.head())
print(f"Shape: {df.shape}")

,data_semana,sku_id,vendas_previsao,vendas_real,data_ingestao,data_processado
0,2025-06-09,SKU_0679,45,42,2026-06-12 12:00:00+00:00,2026-06-18 05:24:37.727931+00:00
1,2025-06-09,SKU_1040,48,44,2026-06-12 12:00:00+00:00,2026-06-18 05:24:37.727931+00:00
2,2025-06-09,SKU_1624,49,47,2026-06-12 12:00:00+00:00,2026-06-18 05:24:37.727931+00:00
3,2025-06-09,SKU_0484,50,51,2026-06-12 12:00:00+00:00,2026-06-18 05:24:37.727931+00:00
4,2025-06-09,SKU_1429,50,51,2026-06-12 12:00:00+00:00,2026-06-18 05:24:37.727931+00:00


Shape: (100000, 6)


## 1.3 Sanity check estrutural

In [9]:
structural_summary = pd.DataFrame({
    "metrica": [
        "linhas",
        "colunas",
        "duplicidades_exatas",
        "memoria_mb",
        "data_min",
        "data_max",
    ],
    "valor": [
        len(df),
        df.shape[1],
        int(df.duplicated().sum()),
        round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        pd.to_datetime(df[DATE_COL], errors="coerce").min() if DATE_COL else None,
        pd.to_datetime(df[DATE_COL], errors="coerce").max() if DATE_COL else None,
    ],
})
display(structural_summary)
display(df.dtypes.rename("dtype").to_frame())

KeyError: 'data'

## 1.4 Profiling automático por coluna

O profiling cria uma visão única para tipos, nulos, cardinalidade, constantes, possíveis erros de tipagem e quantis.

In [ ]:
profile = profile_dataframe(df)
display(profile)

# 2. Data Quality

Este bloco responde: **os dados são tecnicamente confiáveis antes de qualquer feature engineering ou treinamento?**

Falhas com severidade error bloqueiam o treino pelo **Quality Gate**. Warnings exigem investigação/documentação.

## 2.1 Quality checks e gate

In [ ]:
dq_report = run_quality_checks(df, config)
display(dq_report.sort_values(["status", "severidade"], ascending=[True, True]))

approved, critical_failures = quality_gate(dq_report)
print("QUALITY GATE:", "APROVADO" if approved else "BLOQUEADO")

if not approved:
    display(critical_failures)

## 2.2 Cobertura temporal e gaps

Forecast exige frequência temporal consistente. Buracos podem representar ausência real de venda, falha de ingestão ou problema de calendário.

In [ ]:
group_cols = [c for c in ENTITY_KEYS if c in df.columns]

temporal = temporal_summary(
    df,
    DATE_COL,
    group_columns=group_cols,
    freq="D",
)
display(temporal.head(20))

if "periodos_faltantes" in temporal.columns:
    display(
        temporal.sort_values("periodos_faltantes", ascending=False)
        .head(10)
    )

## 2.3 Missingness ao longo do tempo

A taxa global de nulos pode esconder uma quebra recente de coleta. Por isso avaliamos **nulos por variável × período**.

In [ ]:
missing_columns = [c for c in df.columns if c != DATE_COL]
fig, ax, missing_temporal = plot_missingness_over_time(
    df,
    DATE_COL,
    columns=missing_columns,
    freq="M",
)
display(missing_temporal.tail())
fig

## 2.4 Outliers — regra estatística e interpretação de negócio

A visualização principal usa o **boxplot** e a regra de Tukey:

- IQR = Q3 - Q1
- limite inferior = Q1 - 1.5 × IQR
- limite superior = Q3 + 1.5 × IQR

Valores fora dos limites são **outliers estatísticos**, não necessariamente erros.

Em vendas, extremos podem representar promoção, Black Friday, lançamento, feriado ou retorno de estoque. Portanto:

**detectar → investigar → entender contexto → decidir tratamento**

In [ ]:
outlier_columns = [
    col
    for col, rules in config.get("columns", {}).items()
    if rules.get("type") in {"numeric", "number", "float", "integer", "int"}
    and col in df.columns
]

outliers_iqr = outlier_report(
    df,
    columns=outlier_columns,
    method="iqr",
    iqr_multiplier=1.5,
)
display(outliers_iqr.sort_values("pct_outliers", ascending=False))

boxplot_summary = plot_outlier_boxplots(
    df,
    columns=outlier_columns,
    iqr_multiplier=1.5,
)
display(boxplot_summary.sort_values("pct_outliers", ascending=False))

In [ ]:
fig, ax = plot_outlier_rate(outliers_iqr)
fig

### MAD como detector robusto complementar

Também usamos **Median Absolute Deviation (MAD)**. O robust z-score é aproximadamente:

z_robusto = 0.6745 × (x - mediana) / MAD

A regra usada é |z_robusto| > 3.5. Divergências entre IQR e MAD são um sinal para investigar a forma da distribuição.

In [ ]:
outliers_mad = outlier_report(
    df,
    columns=outlier_columns,
    method="mad",
    threshold=3.5,
)
display(outliers_mad.sort_values("pct_outliers", ascending=False))

# 3. Analytical EDA

Aqui a pergunta muda de **“os dados são válidos?”** para **“como o fenômeno de negócio se comporta?”**

Essas análises ajudam a formular hipóteses e orientar feature engineering — sem transformar correlação em causalidade.

## 3.1 Distribuição das variáveis numéricas

Histogramas, média, mediana, skewness, kurtosis, quantis, zeros e negativos ajudam a identificar assimetria, caudas longas, multimodalidade, excesso de zeros e valores impossíveis.

In [ ]:
numeric_summary = plot_numeric_distributions(df)
display(
    numeric_summary.sort_values(
        "skewness",
        key=lambda s: s.abs(),
        ascending=False,
    )
)

## 3.2 Evolução temporal do target

Além da série diária, usamos médias móveis para enxergar tendência e mudanças de regime sem perder completamente a granularidade original.

In [ ]:
fig, ax, target_series = plot_target_time_series(
    df,
    DATE_COL,
    TARGET,
    rolling_windows=(7, 28),
    agg="sum",
)
fig

## 3.3 Sazonalidade

Padrões por dia da semana e mês podem justificar features de calendário, mas devem ser confirmados em períodos suficientemente longos.

In [ ]:
fig, ax, season_week = plot_seasonality(
    df,
    DATE_COL,
    TARGET,
    component="day_of_week",
    agg="mean",
)
display(season_week)
fig

In [ ]:
fig, ax, season_month = plot_seasonality(
    df,
    DATE_COL,
    TARGET,
    component="month",
    agg="mean",
)
display(season_month)
fig

## 3.4 Zeros e negativos no target

Para vendas, zero exige contexto: pode significar ausência de demanda ou **ruptura de estoque**. Venda negativa pode representar devolução ou erro de qualidade.

In [ ]:
target_numeric = pd.to_numeric(df[TARGET], errors="coerce")

zero_negative_summary = pd.DataFrame({
    "metrica": ["vendas_zero", "pct_vendas_zero", "vendas_negativas", "pct_vendas_negativas"],
    "valor": [
        int((target_numeric == 0).sum()),
        float((target_numeric == 0).mean()),
        int((target_numeric < 0).sum()),
        float((target_numeric < 0).mean()),
    ],
})
display(zero_negative_summary)

if "ruptura_estoque" in df.columns:
    display(
        df.groupby("ruptura_estoque")[TARGET]
        .agg(["count", "mean", "median"])
    )

## 3.5 Categorias raras e cardinalidade

Categorias muito raras podem dificultar generalização e encoding. Isso não implica removê-las: primeiro verificamos se representam novos produtos, nichos legítimos ou erro cadastral.

In [ ]:
categorical_candidates = [
    c for c in ["produto_id", "loja_id", "categoria"]
    if c in df.columns
]

rare_categories = rare_category_report(
    df,
    categorical_candidates,
    min_frequency=0.08,
)
display(rare_categories)

## 3.6 Correlação e redundância

Usamos **Spearman** para uma triagem robusta a relações monotônicas. Correlação alta não significa causalidade; correlação quase perfeita com o target também pode indicar **leakage**.

In [ ]:
fig, ax, corr = plot_correlation_heatmap(df, method="spearman")
display(corr)
fig

# 4. ML Readiness

Este bloco responde: **mesmo que os dados sejam válidos e interessantes, eles podem ser usados corretamente no modelo?**

Aqui entram disponibilidade temporal das features, leakage, cold start, drift e integridade de joins.

## 4.1 Feature Availability & Leakage Audit

A pergunta central é:

> **Esta informação estará disponível no instante em que a previsão for feita?**

Uma feature pode ter correlação excelente e ainda ser inutilizável em produção se só existir depois da realização do target.

In [ ]:
feature_catalog = dict(config.get("feature_availability", {}))

if "vendas_confirmadas" in df.columns:
    feature_catalog["vendas_confirmadas"] = {
        "available_at_prediction": False,
        "detail": "Replica o realizado; indisponível no momento da previsão.",
    }

availability_report = feature_availability_audit(
    feature_catalog,
    target=TARGET,
)
display(availability_report)

### Correlações suspeitas com o target

Este detector **não prova leakage**. Ele apenas prioriza variáveis com correlação extremamente alta para auditoria temporal e semântica.

In [ ]:
leakage_candidates = correlation_leakage_candidates(
    df,
    TARGET,
    threshold=0.95,
    method="spearman",
)
display(leakage_candidates)

## 4.2 Cold start / histórico insuficiente

Séries com pouco histórico não devem ser tratadas como se tivessem a mesma previsibilidade das séries maduras.

In [ ]:
cold_start = cold_start_report(
    df,
    ENTITY_KEYS,
    date_column=DATE_COL,
    min_observations=60,
    min_history_days=60,
)
display(cold_start.head(20))

print("Entidades em cold start:", int(cold_start["cold_start"].sum()))

## 4.3 Drift temporal

Comparamos um período histórico contra o período recente via **PSI (Population Stability Index)**.

Heurística de leitura usada no diagnóstico:
- menor que 0.10: drift baixo;
- 0.10 a 0.25: drift moderado;
- maior ou igual a 0.25: drift alto.

PSI é um sinalizador, não uma decisão automática de retreino.

In [ ]:
drift = temporal_drift_report(
    df,
    DATE_COL,
    columns=[c for c in [TARGET, "preco"] if c in df.columns],
    recent_fraction=0.30,
)
display(drift)

## 4.4 Integridade de JOINs / Join explosion

Um JOIN N:N inesperado pode multiplicar linhas e inflar o target silenciosamente.

No case real, o check deve ser executado **antes e depois de cada JOIN relevante**.

In [ ]:
if TABLE_ID is None:
    dim_produto_com_duplicidade = pd.DataFrame({
        "produto_id": ["A", "A", "B", "C"],
        "segmento": ["S1", "S1_DUP", "S2", "S3"],
    })

    rows_before = len(df)
    demo_join = df.merge(dim_produto_com_duplicidade, on="produto_id", how="left")
    rows_after = len(demo_join)

    join_check = join_explosion_check(
        rows_before,
        rows_after,
        max_growth_pct=0.01,
    )
    display(pd.DataFrame([join_check]))
else:
    print("No case real: registrar rows_before e rows_after de cada JOIN relevante.")

## 4.5 Resumo executivo de Model Readiness

A decisão final não é baseada em uma única métrica. Consolidamos riscos técnicos para orientar ações antes da modelagem.

In [ ]:
critical_dq_count = int(
    ((dq_report["status"] == "fail") & (dq_report["severidade"] == "error")).sum()
)

leakage_risk_count = int(
    (availability_report["status"] == "risk").sum()
    + leakage_candidates.get("leakage_candidate", pd.Series(dtype=bool)).sum()
)

cold_start_count = int(cold_start["cold_start"].sum())
high_drift_count = int((drift["nivel_drift"] == "high").sum())

readiness_summary = pd.DataFrame([
    {
        "dimensao": "Data Quality",
        "status": "BLOQUEADO" if critical_dq_count else "OK",
        "riscos": critical_dq_count,
        "acao": "Resolver/justificar falhas críticas antes do treino.",
    },
    {
        "dimensao": "Leakage / disponibilidade",
        "status": "INVESTIGAR" if leakage_risk_count else "OK",
        "riscos": leakage_risk_count,
        "acao": "Validar disponibilidade temporal e semântica das features.",
    },
    {
        "dimensao": "Cold start",
        "status": "INVESTIGAR" if cold_start_count else "OK",
        "riscos": cold_start_count,
        "acao": "Definir estratégia para séries com pouco histórico.",
    },
    {
        "dimensao": "Drift",
        "status": "INVESTIGAR" if high_drift_count else "OK",
        "riscos": high_drift_count,
        "acao": "Entender mudança de regime e impacto no split temporal.",
    },
])

display(readiness_summary)

## Próximos passos para a modelagem

Antes de treinar qualquer modelo:
1. corrigir ou justificar falhas críticas do Quality Gate;
2. definir **target, grão e horizonte** exatos;
3. documentar o **cutoff temporal** de cada feature;
4. remover apenas leakages confirmados — não variáveis só porque correlacionam;
5. definir split/backtest **temporal**, nunca aleatório para forecast;
6. criar um baseline obrigatório;
7. comparar MAE, WAPE, bias e skill vs baseline;
8. segmentar performance por produto/loja, especialmente cold starts;
9. registrar versão dos dados, código e resultados para reprodutibilidade.